In [27]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt

In [28]:
df = pd.read_csv("../../Data/final/environment_data.csv")

In [29]:
df.columns

Index(['City', 'Date', 'PM2.5', 'PM10', 'O3', 'NO2', 'CO', 'SO2',
       'Green_Space', 'Temperature_mean', 'Temperature_max', 'Humidity',
       'Wind_speed', 'Pressure', 'Electricity Consumption', 'isWeekend',
       'Season'],
      dtype='object')

In [30]:
df.drop(["Temperature_max","Electricity Consumption","isWeekend","Date",'PM10', 'O3', 'NO2', 'CO', 'SO2'],axis=1,inplace=True)
df.rename(columns={"PM2.5":"Target"},inplace=True)

In [31]:
from sklearn import pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ["City", "Season"]

num_cols = [
    "Green_Space",
    "Temperature_mean",
    "Humidity",
    "Wind_speed",
    "Pressure",
]

X=df.drop(columns=["Target"])
Y=df["Target"]
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 8, 10],
    'model__min_samples_split':[5, 10, 15],
    'model__min_samples_leaf': [1, 2, 4]
}

preprocessor=ColumnTransformer([
    
    ("num",StandardScaler(),num_cols),
    ("cat",OneHotEncoder(handle_unknown='ignore'),cat_cols)
]
    
)
pipeline=Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
]
)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    error_score='raise',
    return_train_score=True
)

grid_search.fit(X, Y)

best_model = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

print("\nValidation R2 Score:")
print(round(grid_search.best_score_, 4))

print("\nTrain R2 Score:")
print(round(np.max(grid_search.cv_results_['mean_train_score']), 4))

print("\nOverfitting Gap:")
print(round(
    np.max(grid_search.cv_results_['mean_train_score']) - grid_search.best_score_,
    4
))

Best Parameters:
{'model__max_depth': 5, 'model__min_samples_leaf': 4, 'model__min_samples_split': 5, 'model__n_estimators': 100}

Validation R2 Score:
-0.2147

Train R2 Score:
0.9015

Overfitting Gap:
1.1161
